In [2]:
import pandas as pd

In [3]:
%cd ../analysis/3.Clustering/3-2.clustering/

/var2/Works/junhyeong/RXLR_landscape/analysis/3.Clustering/3-2.clustering


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
!ls -al

total 856
drwxrwxr-x. 5 junhyeong junhyeong   4096 Dec 16 20:50 .
drwxrwxr-x. 5 junhyeong junhyeong   4096 Feb 24 10:40 ..
drwxrwxr-x. 2 junhyeong junhyeong   4096 Dec 16 20:39 .ipynb_checkpoints
-rw-rw-r--. 1 junhyeong junhyeong 853201 Dec 16 20:50 RXLR_cluster_domain.tsv
drwxrwxr-x. 6 junhyeong junhyeong   4096 Jan 30 14:04 sequence
drwxrwxr-x. 6 junhyeong junhyeong   4096 Dec 16 20:42 structure


In [5]:
df = pd.read_csv("../../2.Structure_prediction/RXLR_fasta_v3.qc.tsv", sep = "\t")[["entry", "seq"]]

In [6]:
spp = pd.read_csv("../../1.RXLR_prediction/1-2.pattern_match/Predicted_RXLR_effectors.tsv", sep = "\t")[["entry", "spp"]]

In [7]:
df = pd.merge(df, spp, how = "left")

In [8]:
def parse_cluster_seq(df):
    count = df.groupby("rep").count().sort_values(by = "entry", ascending = False)
    count = count.reset_index()

    rep_clust = {}

    for idx in count.index:
        rep = count.loc[idx, "rep"]
        c = count.loc[idx, "entry"]
    
        rep_clust[rep] = idx + 1

    ent_clust = {ent: rep_clust[rep] for rep, ent in zip(df["rep"], df["entry"])}
    

    return ent_clust

In [9]:
def parse_cluster_str(df):
    count = df.groupby("rep").count().sort_values(by = "entry", ascending = False)
    count = count.reset_index()

    rep_clust = {}

    for idx in count.index:
        rep = count.loc[idx, "rep"]
        c = count.loc[idx, "entry"]
    
        if c == 1:
            rep_clust[rep] = -1
    
        else:
            rep_clust[rep] = idx + 1

    ent_clust = {ent: rep_clust[rep] for rep, ent in zip(df["rep"], df["entry"])}
    

    return ent_clust

In [10]:
seq = pd.read_csv("./sequence/RXLR_cluster_v3.tsv", sep = "\t", names = ["rep", "entry"])
seq_cluster = parse_cluster_seq(seq)

In [11]:
strc = pd.read_csv("./structure/foldseek_RXLR_3.tsv", sep = "\t", names = ["rep", "entry"])
str_cluster = parse_cluster_str(strc)

In [12]:
df["seq_cl"] = df["entry"].map(seq_cluster)
df["str_cl"] = df["entry"].map(str_cluster)

In [13]:
df.sort_values(["str_cl", "seq_cl"]).to_csv("RXLR_cluster_domain.tsv", sep = "\t", index = False)

In [22]:
df2 = df[df["str_cl"] != -1]

In [21]:
from sklearn.metrics import homogeneity_completeness_v_measure
import numpy as np

In [23]:
seq_cluster = np.array(df2["seq_cl"])
str_cluster = np.array(df2["str_cl"])

In [24]:
homogeneity_completeness_v_measure(seq_cluster, str_cluster)

(0.7932569863400676, 0.8620422784341927, 0.8262204598776144)

In [25]:
homogeneity_completeness_v_measure(str_cluster, seq_cluster)

(0.8620422784341927, 0.7932569863400676, 0.8262204598776144)